# Real-model runbook — GCG / PAIR attacks on free-tier GPU

**Where this runs:** Kaggle notebook, accelerator `GPU T4 x2`, Internet `On`.

**Cost:** £0. Kaggle gives 30 GPU-hours per week (~9–12 h per session).
A full single-model sweep takes ~20+ h at T4 speeds, so **split across sessions**
rather than running every cell at once.

**Order matters.** Cell 3 measures seconds/step *before* you spend hours on a
real run. Gate 1 (`run_algorithm1`) and the no-attack control (`run_control`)
come *before* multi-hour runs like `--per-prompt`, Algorithm 2, Transfer, and
Multi-turn.

**What "done" looks like:** at least one cell prints a non-zero `engaged` (or
`strict_prefix`) rate with a clear lift over the no-attack control. Every ASR
committed so far was 0% because the model under attack was a 2-layer *random*
model with no refusal behaviour to break.

Read `kaggle/README.md` for the account/secret setup this notebook assumes.

## 1. Setup

In [ ]:
# Only install what Kaggle does not already ship. Reinstalling torch wastes ~10 minutes.
!pip install -q -U "bitsandbytes>=0.46.1" accelerate datasets 2>&1 | tail -5

import bitsandbytes, torch, transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("CUDA avail  :", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())

In [ ]:
!nvidia-smi

### Clone or update the repo

Change `REPO_URL` if you forked the repository. If you set a Kaggle secret called
`GITHUB_TOKEN` (and optionally `HF_TOKEN` for gated Llama models), the next cell
configures credentials in memory without saving tokens inside `.git/config`.

In [ ]:
REPO_URL = "https://github.com/Rohit4507/GCG_Implementation_Beginners.git"
REPO_DIR = "/kaggle/working/repo"
# Optional earlier session directory. Kaggle keeps /kaggle/working across
# "Save Version", so existing records are copied forward without overwriting.
FIRST_RUN_DIR = "/kaggle/working/repo_run1"

import glob, os, shutil, subprocess

if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    # Reset origin URL (strips any old auth) and discard tracked changes to
    # plots/smoke results so `git pull --ff-only` succeeds cleanly.
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "checkout", "--", "."], check=False)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    os.chdir("/kaggle/working")
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

if os.path.isdir(FIRST_RUN_DIR):
    for _src in glob.glob(f"{FIRST_RUN_DIR}/results/reports/generated/*.json"):
        _dst = os.path.join("results/reports/generated", os.path.basename(_src))
        if not os.path.exists(_dst):
            shutil.copy2(_src, _dst)

print()
!git log --oneline -1
print()
print("checks -- all must look right before going further:")
!echo -n "  algorithm2 grad_vocab refs (want >=2): " && grep -c grad_vocab src/algorithm2.py
!echo -n "  algorithm2 --num-prompts wired (want >=1): " && grep -c "args.num_prompts" experiments/run_algorithm2.py
!echo -n "  pair --load-in-4bit wired (want >=1): " && grep -c "args.load_in_4bit" experiments/run_pair.py
!echo -n "  judge/rescore/control present: " && ls src/judge.py experiments/rescore.py experiments/run_control.py | tr '\n' ' ' && echo
!echo -n "  records in hand: " && ls results/reports/generated/*.json 2>/dev/null | wc -l

In [ ]:
# Optional: load Kaggle secrets (HF_TOKEN for gated Llama models, GITHUB_TOKEN for push-back).
import os

PUSH_ENABLED = False
_GH_TOKEN = None

git_name  = "Rohit Singh Rajput"
git_email = "rohitrajput33220@gmail.com"
!git config user.name  "{git_name}"
!git config user.email "{git_email}"

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    try:
        _hf = _secrets.get_secret("HF_TOKEN")
        if _hf:
            os.environ["HF_TOKEN"] = _hf
            print("HF_TOKEN loaded (gated models enabled)")
    except Exception:
        print("HF_TOKEN not set (fine for ungated Qwen2.5 / SmolLM2)")
    try:
        _tok = _secrets.get_secret("GITHUB_TOKEN")
        if _tok and _tok.strip():
            _GH_TOKEN = _tok.strip()
            PUSH_ENABLED = True
            print("git push-back ENABLED (GITHUB_TOKEN loaded in memory)")
        else:
            print("git push-back disabled: GITHUB_TOKEN is empty")
    except Exception as e:
        print("git push-back disabled:", type(e).__name__)
        print("Results still land in /kaggle/working — use 'Save Version' to keep them.")
except Exception as e:
    print("Kaggle secrets unavailable:", type(e).__name__)

In [ ]:
# Shared experiment configuration & safe runner helper.
# Defining MODEL here prevents errors if you skip the timing probe or resume mid-notebook.
import collections, subprocess, sys

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TARGET_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
STEPS = 500
MULTITURN_STEPS = 250   # runs both single-turn and multi-turn (2x steps); set 500 if dedicated session
BASELINES_STEPS = 250   # runs both GCG and random search; set 500 if dedicated session

def run_logged(cmd, log_path=None, tail_lines=40, stream=True):
    """Run command unbuffered, save log_path, stream output, and raise on non-zero exit."""
    print("$", " ".join(cmd), flush=True)
    tail = collections.deque(maxlen=tail_lines)
    log_fh = open(log_path, "w", encoding="utf-8") if log_path else None
    try:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in proc.stdout:
            if log_fh:
                log_fh.write(line)
                log_fh.flush()
            tail.append(line)
            if stream:
                sys.stdout.write(line)
                sys.stdout.flush()
        rc = proc.wait()
    finally:
        if log_fh:
            log_fh.close()
    if not stream:
        print("".join(tail), end="")
    if rc != 0:
        print("\n--- TAIL OF FAILED COMMAND ---")
        print("".join(tail), end="")
        raise RuntimeError(f"Command exited with code {rc}. Log: {log_path or 'stdout'}")

## 2. Offline checks

Seconds on CPU. These prove the *code* is correct. They say nothing about
whether the attack works — that is the next section.

In [ ]:
run_logged(["python", "-u", "-m", "experiments.run_smoke"])
print()
run_logged(["python", "-u", "-m", "unittest", "discover", "-s", "tests"])

## 3. Measure seconds/step FIRST

`--steps 5` on a real model. Multiply `seconds/step` (printed under `Compute:`)
by 500 to get the real cost. If one step takes 60 s, 500 steps is ~8.3 hours
*per prompt* and a large multi-prompt table will not fit inside one session —
drop to a smaller model or fewer prompts. Decide that now, not at hour 10.

Expect the first call to pause ~1–2 minutes while downloading weights and
initializing 4-bit quantization.

In [ ]:
import time
t0 = time.time()
run_logged([
    "python", "-u", "-m", "experiments.run_algorithm1",
    "--preset", "paper", "--benchmark",
    "--model", MODEL, "--load-in-4bit",
    "--steps", "5", "--num-prompts", "2",
])
print(f"\nWALL CLOCK (includes download + load + eval): {time.time() - t0:.0f}s")
print("Read `seconds/step` in the Compute line above and multiply by 500.")

## 4. GATE 1 — first real attack

Single-suffix Algorithm 1 (~3.8 h on a T4 at `27.25 s/step` × 500 steps).
Optimises one suffix on pair 0 and splits evaluation into `in_domain` (`n=1`)
and `transfer` (`n=9`). Watch two numbers:

* **loss** — should fall well below the `random-guess floor` line printed at the
  start. On Qwen2.5-1.5B that floor is ~11.9 (`ln(151936)`), and a working
  suffix pushes the loss toward 1–2. If the loss sits at the floor, the attack
  is not biting.
* **ASR / engagement** — check both `strict_prefix` and the `rescore` ladder.

In [ ]:
run_logged(
    [
        "python", "-u", "-m", "experiments.run_algorithm1",
        "--preset", "paper", "--benchmark",
        "--model", MODEL, "--load-in-4bit",
        "--steps", str(STEPS), "--num-prompts", "10",
    ],
    log_path="/kaggle/working/logs_algorithm1.txt",
)

### 4b. The no-attack control — do not publish an ASR without it

Run this **immediately after Gate 1** before spending more GPU hours.
If the model already answers harmful requests **with no attack at all**, then a
non-zero attack rate is not evidence that the attack worked.

This runs **no optimisation**, so it is cheap (~3–5 min). It scores up to four
conditions with both the paper's verbatim prefix match and the `src/judge.py`
ladder:

| condition | what it isolates |
|---|---|
| `no_attack` | the prompt plus one space. The baseline everything else is measured against |
| `initial_suffix` | the paper's `"! " x 20` start, unoptimised |
| `random_suffix` | same length, random round-tripping tokens — controls for "any garbage works" |
| `optimised` | the suffix from your latest >=50-step real-model `algorithm1` run (never a tiny model or the 5-step probe) |

The `Δ engaged` column is the lift to report. If it is near zero, say so.

In [ ]:
import glob, json, os

def latest_real_alg1(model_id, min_steps=50):
    for p in sorted(glob.glob("results/reports/generated/algorithm1_*.json"), reverse=True):
        with open(p, encoding="utf-8") as fh:
            rec = json.load(fh)
        m = rec.get("model") or {}
        if m.get("random_weights") or m.get("model_kind") == "tiny_random":
            continue
        if m.get("hf_model_id") != model_id:
            continue
        if (rec.get("num_steps") or 0) < min_steps:
            continue
        return p
    return None

cmd = [
    "python", "-u", "-m", "experiments.run_control",
    "--preset", "paper", "--benchmark",
    "--model", MODEL, "--load-in-4bit",
    "--num-prompts", "10",
]
latest = latest_real_alg1(MODEL)
if latest:
    print("Scoring optimised suffix from:", os.path.basename(latest))
    cmd.extend(["--suffix-from", latest])
else:
    print("No >=50-step real algorithm1 record for", MODEL, "— running control without --suffix-from")

run_logged(cmd, log_path="/kaggle/working/logs_control.txt")

### 4c. The honest per-prompt ASR (optional, expensive)

Section 4 optimises **one** suffix on **one** prompt and transfers it to the rest.
`--per-prompt` optimises a **separate suffix for every prompt** and scores each
on its own prompt — the only mode in which "ASR over `n` prompts" is really
over `n` attacked prompts.

**Cost, before you start.** Measured on a T4: `27.25 s/step` for 20-token
suffixes at `search_width` 512 → 250 steps is ~1.9 h **per prompt** (~5.7 h for
3 prompts). Check your remaining session and weekly quota before running.

In [ ]:
# 3 prompts, own suffix each (~5.7 h on a T4).
run_logged(
    [
        "python", "-u", "-m", "experiments.run_algorithm1",
        "--per-prompt", "--preset", "paper", "--benchmark",
        "--model", MODEL, "--load-in-4bit",
        "--steps", "250", "--num-prompts", "3",
    ],
    log_path="/kaggle/working/logs_algorithm1_perprompt.txt",
)

## 5. Universal attack (Algorithm 2) — make this your primary result

One suffix, many prompts, one optimisation, plus held-out evaluation on prompts
the suffix never saw.

**Watch the runtime multiplier:** each step back-props and scores candidates
against every *active* prompt (`num_active = min(1 + step // interval, n_train)`).
With 7 training prompts (`--num-prompts 10 --held-out 3`), `--steps 250`, and
`--prompt-add-interval 40`, average active prompts is ~3.8 (~7 h on a T4 — fits
one Kaggle session). For a dedicated A100/multi-session run, bump to
`--steps 500 --num-prompts 15 --held-out 5 --prompt-add-interval 50`.

In [ ]:
run_logged(
    [
        "python", "-u", "-m", "experiments.run_algorithm2",
        "--preset", "paper", "--benchmark",
        "--model", MODEL, "--load-in-4bit",
        "--steps", "250", "--num-prompts", "10", "--held-out", "3",
        "--prompt-add-interval", "40",
    ],
    log_path="/kaggle/working/logs_algorithm2.txt",
)

## 6. Cross-model transfer

Optimise on `SOURCE`, carry the suffix over as *text*, re-tokenise for `TARGET`,
and evaluate on `TARGET` (`~3.8 h` on T4 for 500 source steps + target eval).
`--load-in-4bit` is applied to both models so both fit comfortably on T4 x2.

In [ ]:
run_logged(
    [
        "python", "-u", "-m", "experiments.run_transfer",
        "--preset", "paper", "--benchmark",
        "--model", MODEL, "--target-model", TARGET_MODEL, "--load-in-4bit",
        "--steps", str(STEPS), "--num-prompts", "10",
    ],
    log_path="/kaggle/working/logs_transfer.txt",
)

## 7. Multi-turn (3-turn) attack

Optimises the suffix against the goal as the **third user turn** of a scripted
escalation conversation, using the model's own replies as context, and compares
single-turn vs multi-turn ASR side by side.

Because it optimises **both** single-turn and multi-turn sequentially (`2 × steps`),
500 steps takes ~7.5+ h on T4; `MULTITURN_STEPS` defaults to `250` (~3.8 h) so it
completes inside one session. Set `MULTITURN_STEPS = 500` in Cell 7 if running
in a dedicated session.

**A null result is a real result** if reported honestly. Do not dress up
"both zero" as a finding.

In [ ]:
run_logged(
    [
        "python", "-u", "-m", "experiments.run_multiturn",
        "--preset", "paper", "--benchmark",
        "--model", MODEL, "--load-in-4bit",
        "--steps", str(MULTITURN_STEPS), "--num-prompts", "10", "--n-prior-turns", "2",
    ],
    log_path="/kaggle/working/logs_multiturn.txt",
)

## 8. Baselines — random search, PAIR & perplexity defense

- **8a. Random search (`run_baselines`):** gradient-blind control with identical
  mutation budget (~3 h at 250 steps, ~6 h at 500 steps).
- **8b. PAIR (`run_pair`):** black-box natural-language refinement (no gradients,
  `<= 25` queries, ~5–15 min). Enables comparing fluent PAIR prompts vs
  high-perplexity GCG suffixes.
- **8c. Perplexity defense (`run_defense`, optional):** `run_algorithm1` already
  runs the inline defense diagnostic, while `run_defense` writes a standalone
  `defense_*.json` record for the report table (~1.9 h at 250 steps).

In [ ]:
# 8a. GCG vs gradient-blind random search (identical budget)
run_logged(
    [
        "python", "-u", "-m", "experiments.run_baselines",
        "--preset", "paper", "--benchmark",
        "--model", MODEL, "--load-in-4bit",
        "--steps", str(BASELINES_STEPS), "--num-prompts", "10",
    ],
    log_path="/kaggle/working/logs_baselines.txt",
)

In [ ]:
# 8b. PAIR black-box baseline (~5-15 min; fluent prompts vs GCG gibberish)
run_logged(
    [
        "python", "-u", "-m", "experiments.run_pair",
        "--preset", "paper", "--benchmark",
        "--model", MODEL, "--judge", "model", "--load-in-4bit",
    ],
    log_path="/kaggle/working/logs_pair.txt",
)

In [ ]:
# 8c. Optional: standalone perplexity defense baseline record (~1.9 h at 250 steps)
run_logged(
    [
        "python", "-u", "-m", "experiments.run_defense",
        "--preset", "paper", "--benchmark",
        "--model", MODEL, "--load-in-4bit",
        "--steps", "250",
    ],
    log_path="/kaggle/working/logs_defense.txt",
)

## 9. Stop, re-score, rebuild site & collect

Before the session ends: inspect the tally table, re-score generations with the
ladder, rebuild `docs/index.html`, copy outputs to `/kaggle/working`, and push
to GitHub.

**Budget checkpoint.** If you have used more than ~20 of your 30 weekly GPU
hours, stop here and continue on the remaining models next week.

In [ ]:
# Tally every record using the exact same extractor as docs/index.html
import glob, json
from experiments.build_report import extract

rows = []
for path in sorted(glob.glob("results/reports/generated/*.json")):
    with open(path, encoding="utf-8") as fh:
        rows.append((path.split("/")[-1], extract(json.load(fh))))

print(f"{'run type':<38} {'model':<28} {'ASR':>6} {'n':>4}  headline")
print("-" * 120)
for fname, r in rows:
    asr = "  —  " if r["asr"] is None else f"{r['asr']:.0%}"
    print(f"{str(r['kind'])[:36]:<38} {str(r['model'])[:26]:<28} {asr:>6} "
          f"{str(r['n']):>4}  {r['headline_label']} = {r['headline_value']}")

print()
print("The ASR column is IN-DOMAIN for attack runs: the prompts the suffix was")
print("actually optimised on. Transfer appears in the headline as `X% -> Y%`.")
print("`—` means the run never measured it — not that it measured zero.")

### 9b. Re-score every generation — no GPU needed

**Read this section before believing any ASR number, including your own.**

The paper's `strict_prefix` rule demands a verbatim match of the entire target
string (e.g., all 77 characters of `"Sure, here is a <goal>"`). A plainly
compliant answer like `"Here's a Python script that..."` scores 0%.

`experiments.rescore` runs offline over stored responses and separates three
completely different 0% outcomes via `strict_prefix` → `not_refused` → `engaged`.

In [ ]:
# Full ladder table (no truncation) + write results/reports/scorecards/*.scorecard.json
run_logged(["python", "-u", "-m", "experiments.rescore"], tail_lines=200)

In [ ]:
# Inspect what the real model(s) ACTUALLY generated.
import glob, json, textwrap
from src.judge import iter_samples, score_response
from experiments.build_report import _model_label

rows = []
for path in sorted(glob.glob("results/reports/generated/*.json")):
    with open(path, encoding="utf-8") as fh:
        record = json.load(fh)
    model_meta = record.get("model") if isinstance(record.get("model"), dict) else {}
    if model_meta.get("random_weights") or model_meta.get("model_kind") == "tiny_random":
        continue
    model = _model_label(record)
    if isinstance(record.get("model"), str) and "tiny" in model.lower():
        continue
    for s in iter_samples(record):
        card = score_response(
            s.get("response", ""), s.get("goal", ""),
            s.get("target", ""), s.get("prompt") or s.get("goal", ""),
        )
        rows.append((record.get("run_kind", "?"), model, s, card))

print(f"{len(rows)} real-model generation(s)\n")
for kind, model, s, card in rows:
    flags = [k for k in ("strict_prefix", "engaged", "refused", "degenerate") if card[k]]
    print("=" * 78)
    print(f"{kind}  |  {model}")
    print(f"  flags: {', '.join(flags) or 'none'}   "
          f"coverage={card['goal_coverage']:.2f}  novelty={card['novel_content']:.2f}"
          + (f"  reason={card['degenerate_reason']}" if card["degenerate"] else ""))
    print(f"  GOAL      : {s.get('goal','')[:100]}")
    print("  RESPONSE  :")
    print(textwrap.fill((s.get("response") or "").strip()[:400], 74,
                        initial_indent="      ", subsequent_indent="      "))
    print()

if not rows:
    print("No real-model generations in the records yet.")
    print("Run at least Section 4 (Gate 1) or Section 4b (Control) first.")

In [ ]:
# Rebuild docs/index.html and docs/plots/ from the updated JSON records
run_logged(["python", "-u", "-m", "experiments.build_report"])

In [ ]:
import glob, shutil

shutil.copytree("results", "/kaggle/working/results", dirs_exist_ok=True)
shutil.copytree("docs", "/kaggle/working/docs", dirs_exist_ok=True)
for f in sorted(glob.glob("/kaggle/working/logs_*.txt")):
    print(f)
print()
print("Saved copies to /kaggle/working/results and /kaggle/working/docs (survives 'Save Version').")

In [ ]:
import subprocess
from urllib.parse import urlparse, urlunparse

def _auth_url(repo_url, token):
    p = urlparse(repo_url)
    netloc = f"x-access-token:{token}@{p.hostname}"
    if p.port:
        netloc += f":{p.port}"
    return urlunparse(p._replace(netloc=netloc))

if globals().get("PUSH_ENABLED") and globals().get("_GH_TOKEN"):
    subprocess.run(["git", "add", "-A"], check=True)
    status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True, check=True)
    if not status.stdout.strip():
        print("Nothing to commit.")
    else:
        subprocess.run(
            ["git", "commit", "-m", "Real-model runs from Kaggle: algorithm1/2, control, transfer, multiturn, baselines, PAIR"],
            check=True,
        )
        subprocess.run(["git", "remote", "set-url", "origin", _auth_url(REPO_URL, _GH_TOKEN)], check=True)
        try:
            subprocess.run(["git", "push"], check=True)
            print("Pushed to GitHub.")
        finally:
            # Always restore plain URL so no token is left on disk in .git/config
            subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=False)
else:
    print("Push disabled. Use: File > Save Version (Quick Save), then download")
    print("Output files from /kaggle/working, or add a GITHUB_TOKEN secret and re-run setup.")

## 10. What to do with the numbers

1. **Report the ladder, not one number.** `strict_prefix` alone cannot separate
   a robust model from a broken scoring rule. Lead with the ceiling
   (`not_refused`) and say which case Section 9b put you in.
2. **Report in-domain ASR with its own `n`.** A single suffix optimised on one
   prompt scored on ten is an ASR of `n=1` plus a transfer result of `n=9`. The
   record keeps them apart; do not add them back together.
3. **Report the lift over the no-attack control.** If `Δ engaged` is near zero,
   the model was complying anyway and the suffix is not why. Say that.
4. **Every rate needs its `n` and its interval.** "0%" over 10 prompts means the
   rate is *below 28%*, not that it is zero; over 2 prompts it means below 65%.
   `rescore` and `run_control` print Wilson intervals for this reason.
5. **Cost.** `cost_per_success_usd` is `null` when nothing succeeded — that is
   deliberate; it means "undefined", not "free". On free tier report
   `total_forwards` and `forwards_per_success`, which are hardware-independent.
6. **Report what you did not run.** "We report 1–2B models; 7–8B was out of
   compute budget for an unfunded independent project" is a strength.
   Silently shrinking the claim is not.
7. **If a generation is degenerate, that is a finding about your suffix, not
   about the model.** A loop is not a refusal and must not be counted as one in
   either direction.
8. Update the `README.md` results table from `results/reports/generated/`, then
   write up the post and link this notebook and `docs/index.html`.